<!-- KERNEL_BANNER -->
> **Use kernel: `mrigi_tor190_v9`**
>
> Set the notebook kernel to *Python (mrigi_tor190_v9)* before running.

# 23k: Open-Ended Recovery of the 7 `_COT` LoRA-adapter models  (ACTIVE)

Companion to 23j. The 7 `_COT` fine-tunes are **LoRA/PEFT adapters** (r=8, α=16, targeting `q_proj`+`v_proj`) — not full HF models — so 23j's `AutoModelForCausalLM.from_pretrained` couldn't load them (missing `config.json`).

**What this notebook does now:**
- Runs on kernel **`mrigi_tor190_v9`** (cloned from v8 + `peft<0.4` installed).
- For each adapter repo, reads `adapter_config.json` to find its base model, loads the base with `AutoModelForCausalLM.from_pretrained(..., local_files_only=True)`, then wraps it with `PeftModel.from_pretrained(base, adapter_repo)`.
- All 7 base models are already in the local HF cache — no downloads.
- Merges into `results_23j_checkpoint.json` so 30b sees a single unified 14-model file.
- Uses `zeolite_openended_100.xlsx` with **Q67 swapped** (original OOM'd on every model at mmr; backup: `zeolite_openended_100_pre_q67swap.xlsx`).

| Adapter | Base |
|---|---|
| `DAPT_LR1e5_COT` | `aleynabeste/AllClassesAbstracts70Mmodel_LR1e5` |
| `synv2V2_step80_COT` | `aleynabeste/model_LR1e5v3_synv2V2_step80` |
| `synv2V2_final_COT` | `aleynabeste/model_LR1e5v3_synv2V2_final` |
| `synv2_base_step80_COT` | `aleynabeste/model_LR1e5v3_synv2_base_step80` |
| `synv2_base_final_COT` | `aleynabeste/model_LR1e5v3_synv2_base_final` |
| `fullpaper_120M_COT` | `aleynabeste/model_LR1e5_fullpaper_longer_120M` |
| `base_llama_COT` | `meta-llama/Meta-Llama-3-8B-Instruct` |

In [1]:
import os
# NOTE: `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` requires torch >= 2.1.
# Kernel `mrigi_tor190_v9` has torch 1.12.1 — do NOT set it.

import torch
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from collections import OrderedDict
import gc
import time as _time

from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# PEFT — needed to load LoRA adapters. peft 0.3.x is torch-1.12 compatible.
import peft
from peft import PeftModel, PeftConfig

with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            key, val = line[len('export '):].split('=', 1)
            os.environ[key] = val.strip('"').strip("'")

hf_token = os.environ.get('HF_TOKEN_BESTE')
if not hf_token:
    raise ValueError('HF_TOKEN_BESTE not found in env.sh')
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
print(f'✓ HF_TOKEN_BESTE set (ending ...{hf_token[-4:]})')
print(f'✓ torch: {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
print(f'✓ peft:  {peft.__version__}')

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


✓ HF_TOKEN_BESTE set (ending ...DOcr)
✓ torch: 2.8.0+cu128  |  CUDA available: True
✓ peft:  0.3.0


In [2]:
!nvidia-smi

Tue Jul 28 19:36:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.119.02             Driver Version: 580.119.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  NVIDIA RTX A5000               Off |   00000000:31:00.0 Off |                  Off |
| 30%   40C    P0             67W /  230W |       4MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   1  NVIDIA RTX A5000               Off |   00000000:4B:00.0 Off |                  Off |
| 30%   39C    P0             66W /  230W |       4MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   2  NVIDIA RTX A5000               Off |   00000000:B1:00.0 Off |                  Off |
| 30%   39C    P0             60W /  230W |       4MiB /  24564MiB |      0%      Default |
|                                         |                        |            

In [3]:
# The 7 COT LoRA adapters. Each name resolves to a PEFT adapter repo;
# the base model is discovered from the adapter\'s adapter_config.json at load time.
MODELS = OrderedDict([
    ('dapt_lr1e5_cot',        {'name': 'aleynabeste/AllClassesAbstracts70Mmodel_LR1e5_COT',      'display_name': 'DAPT_LR1e5_COT',        'gpu': 'cuda:0'}),
    ('synv2V2_step80_cot',    {'name': 'aleynabeste/model_LR1e5v3_synv2V2_step80_COT_FT',        'display_name': 'synv2V2_step80_COT',    'gpu': 'cuda:0'}),
    ('synv2V2_final_cot',     {'name': 'aleynabeste/model_LR1e5v3_synv2V2_final_COT_FT',         'display_name': 'synv2V2_final_COT',     'gpu': 'cuda:0'}),
    ('synv2_base_step80_cot', {'name': 'aleynabeste/model_LR1e5v3_synv2_base_step80_COT_FT',     'display_name': 'synv2_base_step80_COT', 'gpu': 'cuda:0'}),
    ('synv2_base_final_cot',  {'name': 'aleynabeste/model_LR1e5v3_synv2_base_final_COT_FT',      'display_name': 'synv2_base_final_COT',  'gpu': 'cuda:0'}),
    ('fullpaper_120M_cot',    {'name': 'aleynabeste/model_LR1e5_fullpaper_longer_120M_COT_FT',   'display_name': 'fullpaper_120M_COT',    'gpu': 'cuda:0'}),
    ('base_llama_cot',        {'name': 'aleynabeste/base_llama_intruct_COT_FT',                  'display_name': 'base_llama_COT',        'gpu': 'cuda:0'}),
])

K_MMR = 15
MAX_NEW_TOKENS = 400

# Backfill Llama-3 chat template on tokenizers that ship without one (base fine-tunes).
_LLAMA3_CHAT_TEMPLATE = None
def _get_llama3_chat_template():
    global _LLAMA3_CHAT_TEMPLATE
    if _LLAMA3_CHAT_TEMPLATE is None:
        base_tok = AutoTokenizer.from_pretrained(
            'meta-llama/Meta-Llama-3-8B-Instruct',
            trust_remote_code=True, use_fast=True, token=hf_token,
        )
        _LLAMA3_CHAT_TEMPLATE = base_tok.chat_template
    return _LLAMA3_CHAT_TEMPLATE

def _gpu_mem_str(device='cuda:0'):
    a = torch.cuda.memory_allocated(device) / 1e9
    r = torch.cuda.memory_reserved(device) / 1e9
    return f'allocated={a:.2f}GB, reserved={r:.2f}GB'


# ---------------------------------------------------------------------------
# Why we resolve adapter repos to LOCAL snapshot directories:
#
# peft 0.3.x\'s PeftModel.from_pretrained(base, ADAPTER_REPO_ID) calls
# hf_hub_download("adapter_model.bin") which hits the Hub directly and gets a
# 404 — these repos only publish adapter_model.safetensors, and peft 0.3.x
# does not know how to load safetensors adapters. We pre-converted the local
# cached .safetensors → .bin (one-shot conversion done outside the notebook);
# peft will happily load a local .bin, but only if we pass it the DIRECTORY
# path so it never asks the Hub. Hence the _local_snapshot_dir helper below.
# ---------------------------------------------------------------------------
import glob
_HF_HUB_CACHE = os.path.expanduser('~/.cache/huggingface/hub')

def _local_snapshot_dir(repo):
    """Return the newest local snapshot directory for a cached HF repo, or None."""
    d = f"{_HF_HUB_CACHE}/models--{repo.replace('/', '--')}"
    snaps = sorted(glob.glob(f'{d}/snapshots/*'))
    return snaps[-1] if snaps else None


def load_model(model_key):
    """Load a LoRA-adapted model:
       1. Resolve adapter repo → local cache dir (peft can\'t reach Hub for the .bin)
       2. Read adapter_config.json → base_model_name_or_path
       3. Load tokenizer + base model (from local cache)
       4. Wrap base with PeftModel.from_pretrained(base, <adapter_local_dir>)
    """
    cfg_out = MODELS[model_key]
    adapter_repo = cfg_out['name']
    device = cfg_out['gpu']

    print(f"\n{'='*80}")
    print(f"Loading {cfg_out['display_name']}  (adapter: {adapter_repo})")
    print(f"  Before load: {_gpu_mem_str(device)}")
    print(f"{'='*80}")

    # 1) Resolve to local snapshot dir
    adapter_dir = _local_snapshot_dir(adapter_repo)
    if adapter_dir is None:
        raise FileNotFoundError(f'No local snapshot for {adapter_repo} in {_HF_HUB_CACHE}')
    bin_path = f'{adapter_dir}/adapter_model.bin'
    if not os.path.isfile(bin_path):
        raise FileNotFoundError(
            f'{adapter_repo}: missing adapter_model.bin at {bin_path}. '
            f'peft 0.3.x cannot load adapter_model.safetensors — run the one-shot '
            f'.safetensors → .bin conversion first (see zeoRAG/CLAUDE.md).'
        )
    print(f"  → adapter dir: {adapter_dir}")

    # 2) Discover base model from local adapter_config.json
    peft_cfg = PeftConfig.from_pretrained(adapter_dir)
    base_repo = peft_cfg.base_model_name_or_path
    print(f"  → base model:  {base_repo}")

    # 3) Tokenizer from BASE (adapters don\'t ship their own tokenizer files).
    tokenizer = AutoTokenizer.from_pretrained(
        base_repo, trust_remote_code=True, use_fast=True, token=hf_token,
        local_files_only=True,
    )
    if getattr(tokenizer, 'chat_template', None) is None:
        tokenizer.chat_template = _get_llama3_chat_template()
        print(f"  ⚠ tokenizer.chat_template was None — backfilled from Llama-3-8B-Instruct")

    # 4) Base model (already cached; if this fails, load it via 23j on v8 first).
    base = AutoModelForCausalLM.from_pretrained(
        base_repo, device_map=device, trust_remote_code=True,
        torch_dtype=torch.float16, token=hf_token,
        local_files_only=True,
    )

    # 5) Apply the LoRA adapter from LOCAL directory (bypasses hub metadata lookup).
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()

    pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                    max_new_tokens=MAX_NEW_TOKENS, temperature=0.1, do_sample=True)
    llm = HuggingFacePipeline(pipeline=pipe)

    print(f"✓ {cfg_out['display_name']} loaded (base + LoRA) on {device}")
    print(f"  After load:  {_gpu_mem_str(device)}")
    return llm, pipe, model, tokenizer


def cleanup_model(llm, pipe, model, tokenizer, device='cuda:0'):
    print(f"  Before cleanup: {_gpu_mem_str(device)}")
    try: model.to('cpu')
    except Exception as e: print(f"  (model.to('cpu') warn: {str(e)[:80]})")
    try: pipe.model = None; pipe.tokenizer = None
    except Exception: pass
    try: llm.pipeline = None
    except Exception: pass
    del llm; del pipe; del model; del tokenizer
    for _ in range(3):
        gc.collect(); torch.cuda.synchronize(device)
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    print(f"  After cleanup:  {_gpu_mem_str(device)}")
    print('✓ GPU memory cleared')


print(f'✓ {len(MODELS)} LoRA-adapter models configured for recovery')
for k, v in MODELS.items():
    print(f"  {v['display_name']:30s}  {v['name']}")

✓ 7 LoRA-adapter models configured for recovery
  DAPT_LR1e5_COT                  aleynabeste/AllClassesAbstracts70Mmodel_LR1e5_COT
  synv2V2_step80_COT              aleynabeste/model_LR1e5v3_synv2V2_step80_COT_FT
  synv2V2_final_COT               aleynabeste/model_LR1e5v3_synv2V2_final_COT_FT
  synv2_base_step80_COT           aleynabeste/model_LR1e5v3_synv2_base_step80_COT_FT
  synv2_base_final_COT            aleynabeste/model_LR1e5v3_synv2_base_final_COT_FT
  fullpaper_120M_COT              aleynabeste/model_LR1e5_fullpaper_longer_120M_COT_FT
  base_llama_COT                  aleynabeste/base_llama_intruct_COT_FT


In [4]:
# ---- Open-ended prompts and evaluate functions (identical to 23j) ----
OPEN_ENDED_PROMPT = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using the provided context together with your own knowledge. Give a clear, focused answer in 3–5 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Context:
{context}

Question: {question}

Answer:"""

NO_CONTEXT_OPEN_ENDED_PROMPT = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using your own knowledge. Give a clear, focused answer in 3–5 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Question: {question}

Answer:"""

def extract_completion(full_response, tokenizer):
    assistant_tag = '<|start_header_id|>assistant<|end_header_id|>'
    if assistant_tag in full_response:
        completion = full_response.split(assistant_tag)[-1].strip()
    else:
        completion = full_response.strip()
    for token in ['<|eot_id|>', '<|end_of_text|>']:
        completion = completion.replace(token, '').strip()
    return completion

def _clean_open_ended(text):
    t = text.strip()
    for lead in ('Answer:', 'ANSWER:', 'answer:'):
        if t.startswith(lead):
            t = t[len(lead):].strip(); break
    return t

def evaluate_openended_response(llm, query, correct_text, correct_letter, context, tokenizer, meta):
    prompt_text = OPEN_ENDED_PROMPT.format(context=context, question=query)
    formatted_prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}], tokenize=False, add_generation_prompt=True
    )
    try:
        completion = extract_completion(llm(formatted_prompt), tokenizer)
        answer = _clean_open_ended(completion)
        return {'query': query, 'context': context,
                'model_answer': answer or 'INVALID',
                'correct_answer_text': correct_text, 'correct_answer_letter': correct_letter,
                'full_response': completion,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                **({} if answer else {'error': 'Empty completion'})}
    except Exception as e:
        print(f'Processing error: {str(e)[:120]}')
        return {'query': query, 'context': context,
                'model_answer': 'ERROR',
                'correct_answer_text': correct_text, 'correct_answer_letter': correct_letter,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                'error': str(e)}

def evaluate_openended_no_context(llm, query, correct_text, correct_letter, tokenizer, meta):
    prompt_text = NO_CONTEXT_OPEN_ENDED_PROMPT.format(question=query)
    formatted_prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}], tokenize=False, add_generation_prompt=True
    )
    try:
        completion = extract_completion(llm(formatted_prompt), tokenizer)
        answer = _clean_open_ended(completion)
        return {'query': query,
                'model_answer': answer or 'INVALID',
                'correct_answer_text': correct_text, 'correct_answer_letter': correct_letter,
                'full_response': completion,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                **({} if answer else {'error': 'Empty completion'})}
    except Exception as e:
        print(f'Processing error: {str(e)[:120]}')
        return {'query': query,
                'model_answer': 'ERROR',
                'correct_answer_text': correct_text, 'correct_answer_letter': correct_letter,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                'error': str(e)}

def evaluate_method(llm, method_name, questions_df, tokenizer, k=15):
    results = []
    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df),
                       desc=f'Evaluating {method_name} (k={k})'):
        query = row['Question']
        meta = {'title': row.get('Title'), 'doi': row.get('DOI')}
        try:
            if method_name == 'no_context':
                r = evaluate_openended_no_context(llm, query, row['Correct_Answer_Text'],
                                                  row['Correct_Answer_Letter'], tokenizer, meta)
            elif method_name == 'mmr':
                ctx = get_mmr_context(query, k=k)
                r = evaluate_openended_response(llm, query, row['Correct_Answer_Text'],
                                                 row['Correct_Answer_Letter'], ctx, tokenizer, meta)
            else:
                raise ValueError(f'Unknown method: {method_name}')
        except Exception as e:
            print(f'  ⚠ Error on question: {str(e)[:100]}')
            r = {'query': query, 'model_answer': 'ERROR',
                 'correct_answer_text': row['Correct_Answer_Text'],
                 'correct_answer_letter': row['Correct_Answer_Letter'],
                 'title': meta.get('title'), 'doi': meta.get('doi'),
                 'error': str(e)}
        results.append(r)
        gc.collect(); torch.cuda.empty_cache()

    n_valid = sum(1 for r in results if r.get('model_answer') not in ('ERROR', 'INVALID'))
    total = len(results)
    return {'method': method_name, 'k': k,
            'n_valid': n_valid, 'total': total,
            'valid_rate': (n_valid/total*100) if total else 0,
            'detailed_results': results}

In [5]:
from pathlib import Path

embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
INDEX_DIRECTORY = Path('faiss_index')
if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")
vector_db = FAISS.load_local(INDEX_DIRECTORY, embeddings, index_name='index',
                              allow_dangerous_deserialization=True)
print(f'✓ Loaded FAISS index with {vector_db.index.ntotal} vectors')

def get_mmr_context(query, k=15):
    results = vector_db.max_marginal_relevance_search(query, k=k)
    return '\n\n'.join([doc.page_content for doc in results])

print('✓ MMR retrieval function defined')

/tmp/ipykernel_736116/1232108641.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')


✓ Loaded FAISS index with 1474439 vectors
✓ MMR retrieval function defined


In [6]:
# Same dataset as 23j — but Q67 has been swapped from the Pd/ZSM-22 hydroisomerization
# question (which OOM'd on every model at mmr) to the previously-unused 15th replacement row
# (Al organization in SSZ-13). Pre-swap file was backed up to zeolite_openended_100_pre_q67swap.xlsx.
QUESTION_FILE = 'zeolite_openended_100.xlsx'

raw_df = pd.read_excel(QUESTION_FILE, engine='openpyxl')
raw_df = raw_df.dropna(subset=['question', 'correct_answer', 'correct_answer_text'])

questions_df = pd.DataFrame({
    'Question': raw_df['question'].astype(str).str.strip(),
    'Correct_Answer_Text': raw_df['correct_answer_text'].astype(str).str.strip(),
    'Correct_Answer_Letter': raw_df['correct_answer'].astype(str).str.strip().str.upper(),
    'Title': raw_df['title'].astype(str),
    'DOI': raw_df['doi'].astype(str),
}).reset_index(drop=True)

print(f'✓ Loaded {len(questions_df)} questions from {QUESTION_FILE}')
print(f'\n  Q67 (swapped): {questions_df.iloc[67]["Question"][:150]}')

✓ Loaded 100 questions from zeolite_openended_100.xlsx

  Q67 (swapped): In Si-rich SSZ-13, why can some framework Al arrangements fail to generate exchange sites for bare divalent cations (e.g., Co2+) yet still accommodate


In [7]:
# Merge into the existing 23j checkpoint so 30b sees a unified 14-model file.
CHECKPOINT_PATH = 'results_23j_checkpoint.json'
MAX_CONSECUTIVE_LOAD_FAILS = 2

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_PATH = f'results_23k_complete_{timestamp}.json'
PROGRESS_LOG = f'results_23k_progress_{timestamp}.log'

def _is_model_complete(res, expected_total):
    return (isinstance(res, dict)
            and 'no_context' in res and 'mmr' in res
            and res['no_context'].get('total', 0) == expected_total
            and res['mmr'].get('total', 0) == expected_total)

def _log(msg):
    stamped = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(stamped, flush=True)
    with open(PROGRESS_LOG, 'a') as f:
        f.write(stamped + '\n')

def _save_checkpoint(all_results, tag):
    for path in (CHECKPOINT_PATH, SAVE_PATH):
        with open(path, 'w') as f:
            json.dump(all_results, f, indent=2, default=str)
    sz = os.path.getsize(SAVE_PATH) / 1024
    _log(f'   💾 checkpoint saved [{tag}]: {SAVE_PATH} ({sz:.0f} KB)  + {CHECKPOINT_PATH}')

EXPECTED_TOTAL = len(questions_df)
_log(f'Expected questions per model: {EXPECTED_TOTAL}')
_log(f'Progress log: {PROGRESS_LOG}')

# ALWAYS resume — this notebook is by definition merging into a prior checkpoint.
if not os.path.exists(CHECKPOINT_PATH):
    _log(f'⚠ No existing {CHECKPOINT_PATH} found — starting fresh (23j does not appear to have run)')
    all_results = {}
else:
    with open(CHECKPOINT_PATH) as f:
        all_results = json.load(f)
    done_all = [k for k, v in all_results.items() if _is_model_complete(v, EXPECTED_TOTAL)]
    _log(f'✓ Loaded existing checkpoint — {len(done_all)} models already complete overall')
    for d in done_all:
        _log(f'    ✓ {d}')

# ------------------------------------------------------------------
# Model loop — only the 7 recovery models
# ------------------------------------------------------------------
run_start = _time.time()
consecutive_load_fails = 0
model_items = list(MODELS.items())
n_total = len(model_items)

for m_idx, (model_key, cfg) in enumerate(model_items, start=1):
    display_name = cfg['display_name']

    if _is_model_complete(all_results.get(display_name), EXPECTED_TOTAL):
        _log(f'[{m_idx}/{n_total}] SKIP  {display_name} — already complete in checkpoint')
        continue

    model_start = _time.time()
    _log('')
    _log('#' * 80)
    _log(f'[{m_idx}/{n_total}] MODEL: {display_name}  (elapsed so far: {(model_start-run_start)/60:.1f} min)')
    _log('#' * 80)

    try:
        llm, pipe, model, tokenizer = load_model(model_key)
        consecutive_load_fails = 0
    except Exception as e:
        consecutive_load_fails += 1
        _log(f'✗ Failed to load {display_name}: {str(e)[:200]}')
        all_results[display_name] = {'load_error': str(e)}
        _save_checkpoint(all_results, tag=f'load_fail:{display_name}')
        if consecutive_load_fails >= MAX_CONSECUTIVE_LOAD_FAILS:
            _log('')
            _log('!' * 80)
            _log(f'ABORTING: {consecutive_load_fails} consecutive load failures — systemic (auth, disk, network?)')
            _log('!' * 80)
            break
        continue

    all_results[display_name] = {}
    for method in ['no_context', 'mmr']:
        method_start = _time.time()
        _log(f'--- [{display_name}] {method}{" (k=" + str(K_MMR) + ")" if method == "mmr" else ""} ---')
        try:
            result = evaluate_method(llm, method, questions_df, tokenizer, k=K_MMR)
            all_results[display_name][method] = result
            dt = (_time.time() - method_start) / 60
            _log(f'✓ {method}: {result["n_valid"]}/{result["total"]} valid  ({dt:.1f} min)')
        except Exception as e:
            _log(f'✗ {method} error: {str(e)[:200]}')
            all_results[display_name][method] = {'error': str(e), 'n_valid': 0, 'total': 0, 'valid_rate': 0}
        _save_checkpoint(all_results, tag=f'{display_name}:{method}')

    cleanup_model(llm, pipe, model, tokenizer)
    dt_model = (_time.time() - model_start) / 60
    dt_total = (_time.time() - run_start) / 60
    _log(f'⏱  {display_name} done in {dt_model:.1f} min  (total: {dt_total:.1f} min)')

_log('')
_log('=' * 80)
_log(f'23k recovery complete. Total wall-clock: {(_time.time()-run_start)/60:.1f} min')
_log(f'Merged checkpoint: {CHECKPOINT_PATH}')
_log(f'This-run snapshot: {SAVE_PATH}')
_log(f'Progress log:      {PROGRESS_LOG}')
_log('=' * 80)

[19:36:51] Expected questions per model: 100
[19:36:51] Progress log: results_23k_progress_20260728_193651.log
[19:36:51] ✓ Loaded existing checkpoint — 7 models already complete overall
[19:36:51]     ✓ Llama-3-8B-Instruct
[19:36:51]     ✓ DAPT_LR1e5
[19:36:51]     ✓ synv2V2_step80
[19:36:51]     ✓ synv2V2_final
[19:36:51]     ✓ synv2_base_step80
[19:36:51]     ✓ synv2_base_final
[19:36:51]     ✓ fullpaper_120M_LR1e5
[19:36:51] 
[19:36:51] ################################################################################
[19:36:51] [1/7] MODEL: DAPT_LR1e5_COT  (elapsed so far: 0.0 min)
[19:36:51] ################################################################################

Loading DAPT_LR1e5_COT  (adapter: aleynabeste/AllClassesAbstracts70Mmodel_LR1e5_COT)
  Before load: allocated=0.00GB, reserved=0.00GB
  → adapter dir: /home/synthesisproject/.cache/huggingface/hub/models--aleynabeste--AllClassesAbstracts70Mmodel_LR1e5_COT/snapshots/a315bcdbd18906085387d64acc08f0210ef95fa6
  → base

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertFo

✓ DAPT_LR1e5_COT loaded (base + LoRA) on cuda:0
  After load:  allocated=16.21GB, reserved=16.22GB
[19:37:13] --- [DAPT_LR1e5_COT] no_context ---


...

[22:25:19] ✓ no_context: 100/100 valid  (10.9 min)


[22:25:20]    💾 checkpoint saved [base_llama_COT:no_context]: results_23k_complete_20260728_193651.json (61487 KB)  + results_23j_checkpoint.json
[22:25:20] --- [base_llama_COT] mmr (k=15) ---


Evaluating mmr (k=15):   0%|          | 0/100 [00:00<?, ?it/s]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Evaluating mmr (k=15):   1%|          | 1/100 [00:14<23:37, 14.32s/it]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: Y

[22:39:02] ✓ mmr: 100/100 valid  (13.7 min)


[22:39:03]    💾 checkpoint saved [base_llama_COT:mmr]: results_23k_complete_20260728_193651.json (65928 KB)  + results_23j_checkpoint.json
  Before cleanup: allocated=16.31GB, reserved=16.32GB
  After cleanup:  allocated=0.10GB, reserved=0.11GB
✓ GPU memory cleared
[22:39:09] ⏱  base_llama_COT done in 24.9 min  (total: 182.3 min)
[22:39:09] 
[22:39:09] ================================================================================
[22:39:09] 23k recovery complete. Total wall-clock: 182.3 min
[22:39:09] Merged checkpoint: results_23j_checkpoint.json
[22:39:09] This-run snapshot: results_23k_complete_20260728_193651.json
[22:39:09] Progress log:      results_23k_progress_20260728_193651.log
[22:39:09] ================================================================================


In [8]:
# Sanity summary across the recovered models
print('\n' + '='*90)
print(f'23k recovery summary — n={EXPECTED_TOTAL}')
print('='*90)
print(f'{"Model":<32} {"no_context (valid)":>22} {"mmr_k15 (valid)":>22}')
print('-'*90)
for cfg in MODELS.values():
    dn = cfg['display_name']
    res = all_results.get(dn, {})
    if 'load_error' in res:
        print(f'{dn:<32} {"LOAD ERROR":>22}')
        continue
    nc = res.get('no_context', {})
    mm = res.get('mmr', {})
    nc_str = str(nc.get('n_valid', 0)) + '/' + str(nc.get('total', 0))
    mm_str = str(mm.get('n_valid', 0)) + '/' + str(mm.get('total', 0))
    print(f'{dn:<32} {nc_str:>22} {mm_str:>22}')
print('='*90)
print('Full 14-model results now live in results_23j_checkpoint.json — feed to 30b.')


23k recovery summary — n=100
Model                                no_context (valid)        mmr_k15 (valid)
------------------------------------------------------------------------------------------
DAPT_LR1e5_COT                                  100/100                100/100
synv2V2_step80_COT                              100/100                100/100
synv2V2_final_COT                               100/100                100/100
synv2_base_step80_COT                           100/100                100/100
synv2_base_final_COT                            100/100                100/100
fullpaper_120M_COT                              100/100                100/100
base_llama_COT                                  100/100                100/100
Full 14-model results now live in results_23j_checkpoint.json — feed to 30b.
